# Weather Data Producer — Open-Meteo API

Polls [Open-Meteo](https://open-meteo.com) (free, no API key) for current weather conditions across **15 cities** worldwide and writes JSON files to a UC volume for streaming ingestion.

**Pipeline**: Producer → Bronze (Auto Loader) → Silver (Dedup + Type Cast) → Gold (KPI Aggregation) → Dashboard

In [0]:
import requests, json, uuid, time
from datetime import datetime, timezone

API_URL = "https://api.open-meteo.com/v1/forecast"
VOLUME_PATH = "/Volumes/workspace/landing/weather_landing"

CITIES = {
    "Bengaluru": (12.97, 77.59), "Hyderabad": (17.38, 78.49), "Chennai": (13.08, 80.27),
    "Mumbai": (19.08, 72.88), "Delhi": (28.61, 77.21), "Kolkata": (22.57, 88.36),
    "Pune": (18.52, 73.86), "Ahmedabad": (23.02, 72.57), "Jaipur": (26.91, 75.79),
    "Kochi": (9.93, 76.27), "London": (51.51, -0.13), "New York": (40.71, -74.01),
    "Tokyo": (35.68, 139.69), "Sydney": (-33.87, 151.21), "Dubai": (25.20, 55.27),
}
CURRENT_VARS = [
    "temperature_2m", "relative_humidity_2m", "apparent_temperature", "precipitation",
    "rain", "weather_code", "cloud_cover", "pressure_msl", "wind_speed_10m",
    "wind_direction_10m", "wind_gusts_10m", "is_day",
]

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.landing.weather_landing")

def fetch_weather():
    """One HTTP call for all cities (Open-Meteo accepts comma-separated coordinates)."""
    names = list(CITIES)
    params = {
        "latitude": ",".join(str(CITIES[n][0]) for n in names),
        "longitude": ",".join(str(CITIES[n][1]) for n in names),
        "current": ",".join(CURRENT_VARS),
        "timezone": "GMT",
    }
    resp = requests.get(API_URL, params=params, timeout=30)
    resp.raise_for_status()
    payload = resp.json()
    payload = payload if isinstance(payload, list) else [payload]
    polled_at = datetime.now(timezone.utc).isoformat()
    events = []
    for name, loc in zip(names, payload):
        cur = loc.get("current", {})
        events.append({
            "event_id": f"{name}|{cur.get('time')}",
            "city": name,
            "latitude": loc.get("latitude"),
            "longitude": loc.get("longitude"),
            "elevation": loc.get("elevation"),
            "observation_time_utc": cur.get("time"),
            "polled_at_utc": polled_at,
            **{k: cur.get(k) for k in CURRENT_VARS},
        })
    return events

In [0]:
events = fetch_weather()
file_name = f"weather_{int(time.time())}_{uuid.uuid4().hex[:8]}.json"
file_path = f"{VOLUME_PATH}/{file_name}"
content = "\n".join(json.dumps(e) for e in events)
dbutils.fs.put(file_path, content, overwrite=True)
print(f"✅ Wrote {len(events)} weather readings to {file_path}")
print(f"   Cities: {', '.join(e['city'] for e in events)}")